# ArtaMatch — the tradition sweep

Every astrological tradition that could be implemented, encoded as feature blocks, scored against a
marriage outcome, and then ensembled. Fifteen tradition modules, 211 feature blocks, 59,428 columns.

The substrate arrives as a dataset; this notebook only runs it. Nothing here is fitted to the test rows:
splits are person-disjoint (a person who appears in two marriages is never on both sides), every score is
a mean over many splits, and the stack's meta-learner sees only out-of-fold base predictions.

In [ ]:
import os, sys, subprocess, time, json
T0 = time.time()
# WHAT HARDWARE DID WE ACTUALLY GET? A GPU kernel that silently falls back to CPU would look like a slow
# run rather than a misconfiguration, so this is printed before anything else.
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
          or "no GPU reported by nvidia-smi")
except FileNotFoundError:
    print("nvidia-smi not present — this is a CPU session")
# pyswisseph is the one dependency Kaggle does not ship. Three tradition modules call it directly.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyswisseph==2.10.3.2"], check=True)
import swisseph as swe
print("pyswisseph", swe.version)
for m in ("xgboost", "lightgbm", "catboost"):
    try:
        mod = __import__(m); print(m, mod.__version__)
    except Exception as e:
        print(m, "MISSING", str(e)[:60])
print(f"{os.cpu_count()} cores")

In [ ]:
# WHICH BENCHMARK — set by the pusher, so one notebook serves both.
COUPLES_FILE = "couples-parents.json"
EPHEM_FILE = "ephem-par2.npz"
print("benchmark:", COUPLES_FILE)

In [ ]:
# Find the payload by looking for a MARKER FILE rather than guessing the mount path. The first run
# failed on an assert here: Kaggle now nests dataset mounts, so /kaggle/input held only ['datasets']
# and the flat directory was further down. Searching for core.py is immune to the layout changing again.
SRC = None
for root, dirs, files in os.walk("/kaggle/input"):
    # Marker files: core.py AND evalx.py, both of which are always shipped. This previously looked for
    # ephem-cache.npz, which stopped being shipped when the payload was slimmed — so the finder matched
    # nothing and the run died with "payload not found" while the payload was sitting right there.
    if "core.py" in files and "evalx.py" in files:
        SRC = root
        break
assert SRC, f"payload not found under /kaggle/input; tree top = {os.listdir('/kaggle/input')}"
print("payload at", SRC)
print(len(os.listdir(SRC)), "files:", ", ".join(sorted(os.listdir(SRC))[:6]), "...")

WORK = "/kaggle/working"
os.makedirs(f"{WORK}/blocks", exist_ok=True)
os.environ["AQ_EPHE"] = SRC                      # the .se1 files sit flat beside the code
# The ephemeris cache is a BUILD ARTEFACT and always goes to the writable directory. Pointing it at the
# dataset made core.py try to write into /kaggle/input, which is read-only: "OSError [Errno 30]". The
# caches are not shipped anyway — they are regenerated here from the .se1 files in about two minutes.
os.environ["AQ_EPHEM_CACHE"] = f"{WORK}/{EPHEM_FILE or 'ephem-computed.npz'}"
COUPLES_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if COUPLES_FILE in files:
        cand = os.path.join(root, COUPLES_FILE)
        if COUPLES_PATH is None or os.path.getsize(cand) > os.path.getsize(COUPLES_PATH):
            COUPLES_PATH = cand          # prefer the larger file: the updated dataset is the bigger one
assert COUPLES_PATH, f"{COUPLES_FILE} not found under /kaggle/input"
print("couples:", COUPLES_PATH, os.path.getsize(COUPLES_PATH), "bytes")
os.environ["AQ_COUPLES"] = COUPLES_PATH
os.environ["AQ_BLOCKS"] = f"{WORK}/blocks"
os.environ["AQ_OUTDIR"] = WORK
os.environ["AQ_WORKERS"] = str(max(1, os.cpu_count() or 4))
# THE FOUR-INPUT CONTRACT, enforced by configuration rather than by remembering not to select a block.
# The model may see only each partner's date of birth and place of birth. The nationality module carries
# citizenship and sex, which a page cannot collect, so it is excluded from both the module list and the
# context arm; geo4 supplies place as two real coordinates instead of a country one-hot.
if COUPLES_FILE == "couples-parents.json":
    os.environ["AQ_ONLY"] = ("babylonian_egyptian,chinese,harmonics,hellenistic,lunar_calendrical,"
                             "mesoamerican,modern_western,persian_arabic,tibetan_seasia,uranian,"
                             "vedic_core,vedic_match,geo4,precision,cohort")
    os.environ["AQ_CONTEXT_KEYS"] = ("geo4::geo: EVERYTHING,precision::prec: EVERYTHING,"
                                     "cohort::coh: EVERYTHING")
    print("four-input contract enforced: no citizenship, no sex, place as coordinates")
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"
sys.path.insert(0, SRC)
from core import load
E = load()
print(f"{E.n:,} couples, {int(E.Y.sum()):,} per class, {len(set(E.gid.tolist())):,} person groups")

In [ ]:
import run, evalx
# Point every booster at the GPU when there is one. xgboost takes device="cuda"; lightgbm and catboost
# need their own flags, and catboost's GPU build ignores some CPU-only options, so each is set separately
# rather than assuming one switch covers all three.
try:
    import xgboost, subprocess
    HAS_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    HAS_GPU = False
print("GPU available:", HAS_GPU)
if HAS_GPU:
    _orig = evalx._boosters
    def _gpu_boosters(nfeat):
        out = _orig(nfeat)
        try:
            from xgboost import XGBClassifier
            out["xgboost"] = lambda: XGBClassifier(
                n_estimators=evalx._n(600), learning_rate=0.03, max_depth=5, subsample=0.8,
                colsample_bytree=0.6, reg_lambda=2.0, min_child_weight=4, n_jobs=-1,
                tree_method="hist", device="cuda", eval_metric="logloss", random_state=0)
            out["xgboost deep"] = lambda: XGBClassifier(
                n_estimators=evalx._n(1200), learning_rate=0.015, max_depth=8, subsample=0.7,
                colsample_bytree=0.4, reg_lambda=5.0, min_child_weight=8, n_jobs=-1,
                tree_method="hist", device="cuda", eval_metric="logloss", random_state=0)
        except Exception as e:
            print("xgboost GPU setup failed:", str(e)[:80])
        try:
            from catboost import CatBoostClassifier
            out["catboost"] = lambda: CatBoostClassifier(
                iterations=evalx._n(800), learning_rate=0.03, depth=6, l2_leaf_reg=6.0,
                task_type="GPU", devices="0", verbose=0, allow_writing_files=False, random_seed=0)
        except Exception as e:
            print("catboost GPU setup failed:", str(e)[:80])
        return out
    evalx._boosters = _gpu_boosters
    # one worker only: the GPU is the bottleneck and six processes would just queue on it
    run.WORKERS = 1
from evalx import MODELS
print(f"{len(MODELS(100))} models in the zoo:")
for m in MODELS(100):
    print("   ", m)
run.collect()

In [ ]:
# Screen every block, then sweep the survivors across representations and the full zoo.
run.SCREEN_MODELS = ["logistic L2 (C=0.1)", "hist gradient boosting", "extra trees"]
run.SCREEN_SPLITS = 8
run.screen()

In [ ]:
run.KEEP_FOR_DEEP = 70
run.DEEP_SPLITS = 20
# Everything the environment has, boosters included — this is the point of running here.
run.DEEP_MODELS = [m for m in MODELS(100)]
run.reps_for = lambda ncol: (["raw"] + (["topk", "pca", "quantile"] if ncol > 8 else [])
                             + (["rff"] if ncol <= 512 else []) + (["inter"] if ncol <= 22 else []))
run.deep()

In [ ]:
run.KEEP_FOR_OOF = 120
run.oof()
run.stack()

In [ ]:
# One bundle to pull back.
import shutil, json, os
os.makedirs("/kaggle/working/bundle", exist_ok=True)
for f in ("manifest.json", "screen.json", "deep.json", "stack.json", "oof.npz"):
    p = f"/kaggle/working/{f}"
    if os.path.exists(p):
        shutil.copy(p, f"/kaggle/working/bundle/{f}")
        print(f, os.path.getsize(p), "bytes")
print(f"total {time.time()-T0:.0f}s")